# Setup path

In [1]:
import os
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

import pywt


# ============================================================
# 1. DATASET PATH
# ============================================================

DATASET_PATH = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET"
)


# ============================================================
# 2. SAVED TUMOR ROI DATASET
# ============================================================

ROI_DATASET_PATH = (
    DATASET_PATH / "Tumor_ROI_Morphology"
)


# ============================================================
# 3. TRAIN AND TEST PATHS
# ============================================================

ROI_TRAIN_PATH = (
    ROI_DATASET_PATH / "train"
)

ROI_TEST_PATH = (
    ROI_DATASET_PATH / "test"
)


# ============================================================
# 4. FEATURE OUTPUT DIRECTORY
# ============================================================

FEATURE_PATH = (
    DATASET_PATH / "Features_DWT"
)

FEATURE_PATH.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. DWT PARAMETERS
# ============================================================

# Wavelet used for decomposition
DWT_WAVELET = "haar"

# Number of decomposition levels
DWT_LEVEL = 1


# ============================================================
# 6. DWT SUB-BANDS
# ============================================================

DWT_SUBBANDS = [
    "LL",
    "LH",
    "HL",
    "HH"
]


# ============================================================
# 7. DWT STATISTICAL FEATURES
# ============================================================

DWT_FEATURES = [
    "mean",
    "std",
    "variance",
    "energy",
    "entropy"
]


# ============================================================
# 8. DISPLAY CONFIGURATION
# ============================================================

print("=" * 60)
print("DWT FEATURE EXTRACTION SETUP")
print("=" * 60)

print("\nROI Dataset:")
print(ROI_DATASET_PATH)

print("\nTrain ROI:")
print(ROI_TRAIN_PATH)

print("\nTest ROI:")
print(ROI_TEST_PATH)

print("\nFeature Output:")
print(FEATURE_PATH)

print("\nDWT Wavelet:")
print(DWT_WAVELET)

print("\nDWT Decomposition Level:")
print(DWT_LEVEL)

print("\nDWT Sub-bands:")
print(DWT_SUBBANDS)

print("\nDWT Statistical Features:")
print(DWT_FEATURES)

DWT FEATURE EXTRACTION SETUP

ROI Dataset:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI_Morphology

Train ROI:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI_Morphology\train

Test ROI:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI_Morphology\test

Feature Output:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT

DWT Wavelet:
haar

DWT Decomposition Level:
1

DWT Sub-bands:
['LL', 'LH', 'HL', 'HH']

DWT Statistical Features:
['mean', 'std', 'variance', 'energy', 'entropy']


 # ROI CROPPING + DWT FEATURE EXTRACTION FUNCTIONS

In [2]:
# ============================================================
# CELL 2: ROI CROPPING + DWT FEATURE EXTRACTION FUNCTIONS
# ============================================================

import numpy as np
import cv2
import pywt


# ============================================================
# 1. CROP TUMOR ROI
# ============================================================

def crop_tumor_roi(roi, padding=5):
    """
    Crop the saved tumor ROI to the bounding box
    containing non-zero tumor pixels.

    A small padding is added around the tumor boundary
    to preserve surrounding boundary information.
    """

    # --------------------------------------------------------
    # Find non-zero pixels
    # --------------------------------------------------------

    y_coords, x_coords = np.where(
        roi > 0
    )

    # --------------------------------------------------------
    # If ROI is completely black
    # --------------------------------------------------------

    if len(x_coords) == 0:
        return None

    # --------------------------------------------------------
    # Find bounding box
    # --------------------------------------------------------

    x_min = np.min(x_coords)
    x_max = np.max(x_coords)

    y_min = np.min(y_coords)
    y_max = np.max(y_coords)

    # --------------------------------------------------------
    # Add padding
    # --------------------------------------------------------

    x_min = max(
        0,
        x_min - padding
    )

    x_max = min(
        roi.shape[1] - 1,
        x_max + padding
    )

    y_min = max(
        0,
        y_min - padding
    )

    y_max = min(
        roi.shape[0] - 1,
        y_max + padding
    )

    # --------------------------------------------------------
    # Crop
    # --------------------------------------------------------

    cropped_roi = roi[
        y_min:y_max + 1,
        x_min:x_max + 1
    ]

    return cropped_roi


# ============================================================
# 2. PREPARE ROI FOR DWT
# ============================================================

def prepare_for_dwt(roi):
    """
    Prepare the cropped tumor ROI for DWT.

    Steps:
        1. Convert to grayscale if required
        2. Convert to float32
        3. Normalize intensity to 0-1
    """

    # --------------------------------------------------------
    # Convert to grayscale if image has 3 channels
    # --------------------------------------------------------

    if len(roi.shape) == 3:

        roi = cv2.cvtColor(
            roi,
            cv2.COLOR_BGR2GRAY
        )

    # --------------------------------------------------------
    # Convert to float32
    # --------------------------------------------------------

    roi = roi.astype(
        np.float32
    )

    # --------------------------------------------------------
    # Handle completely empty ROI
    # --------------------------------------------------------

    if np.max(roi) == 0:

        return None

    # --------------------------------------------------------
    # Normalize ROI to 0-1
    # --------------------------------------------------------

    roi = cv2.normalize(
        roi,
        None,
        0.0,
        1.0,
        cv2.NORM_MINMAX
    )

    return roi


# ============================================================
# 3. CALCULATE DWT STATISTICAL FEATURES
# ============================================================

def calculate_dwt_statistics(coefficients):
    """
    Calculate statistical features from one DWT sub-band.

    Features:
        Mean
        Standard deviation
        Variance
        Energy
        Entropy
    """

    coefficients = coefficients.astype(
        np.float32
    )

    # --------------------------------------------------------
    # Mean
    # --------------------------------------------------------

    mean = np.mean(
        coefficients
    )

    # --------------------------------------------------------
    # Standard deviation
    # --------------------------------------------------------

    std = np.std(
        coefficients
    )

    # --------------------------------------------------------
    # Variance
    # --------------------------------------------------------

    variance = np.var(
        coefficients
    )

    # --------------------------------------------------------
    # Energy
    # --------------------------------------------------------

    energy = np.sum(
        coefficients ** 2
    )

    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------

    absolute_coefficients = np.abs(
        coefficients
    )

    total = np.sum(
        absolute_coefficients
    )

    if total == 0:

        entropy = 0.0

    else:

        probabilities = (
            absolute_coefficients / total
        )

        # Remove zero probabilities
        probabilities = probabilities[
            probabilities > 0
        ]

        entropy = -np.sum(
            probabilities *
            np.log2(probabilities)
        )

    return [
        mean,
        std,
        variance,
        energy,
        entropy
    ]


# ============================================================
# 4. EXTRACT DWT FEATURES
# ============================================================

def extract_dwt_features(roi):
    """
    Extract DWT-based features from a tumor ROI.

    DWT configuration:
        Wavelet  : Haar
        Level    : 1

    Sub-bands:
        LL
        LH
        HL
        HH

    Features per sub-band:
        Mean
        Standard deviation
        Variance
        Energy
        Entropy

    Total:
        4 sub-bands × 5 features = 20 features
    """

    # --------------------------------------------------------
    # Crop black background
    # --------------------------------------------------------

    cropped_roi = crop_tumor_roi(
        roi,
        padding=5
    )

    # --------------------------------------------------------
    # Handle empty ROI
    # --------------------------------------------------------

    if cropped_roi is None:

        return None

    # --------------------------------------------------------
    # Prepare ROI
    # --------------------------------------------------------

    prepared_roi = prepare_for_dwt(
        cropped_roi
    )

    if prepared_roi is None:

        return None

    # --------------------------------------------------------
    # Apply 2D Discrete Wavelet Transform
    # --------------------------------------------------------

    LL, (LH, HL, HH) = pywt.dwt2(
        prepared_roi,
        DWT_WAVELET
    )

    # --------------------------------------------------------
    # Store DWT sub-bands
    # --------------------------------------------------------

    subbands = {
        "LL": LL,
        "LH": LH,
        "HL": HL,
        "HH": HH
    }

    # --------------------------------------------------------
    # Extract statistical features
    # --------------------------------------------------------

    features = []

    for subband_name in DWT_SUBBANDS:

        coefficients = subbands[
            subband_name
        ]

        statistics = (
            calculate_dwt_statistics(
                coefficients
            )
        )

        features.extend(
            statistics
        )

    # --------------------------------------------------------
    # Return feature vector
    # --------------------------------------------------------

    return np.array(
        features,
        dtype=np.float32
    )


# ============================================================
# 5. CREATE DWT FEATURE COLUMN NAMES
# ============================================================

def create_feature_names():

    feature_names = []

    for subband in DWT_SUBBANDS:

        for feature in DWT_FEATURES:

            feature_name = (
                f"DWT_{subband}_{feature}"
            )

            feature_names.append(
                feature_name
            )

    return feature_names


# ============================================================
# 6. CREATE FEATURE NAMES
# ============================================================

FEATURE_NAMES = (
    create_feature_names()
)


# ============================================================
# 7. DISPLAY INFORMATION
# ============================================================

print("=" * 60)
print("DWT FUNCTIONS READY")
print("=" * 60)

print("\nWavelet:")
print(DWT_WAVELET)

print("\nDecomposition Level:")
print(DWT_LEVEL)

print("\nSub-bands:")
print(DWT_SUBBANDS)

print("\nFeatures per sub-band:")
print(DWT_FEATURES)

print("\nTotal DWT features:")
print(len(FEATURE_NAMES))

print("\nFeature names:")

for name in FEATURE_NAMES:

    print(" -", name)

DWT FUNCTIONS READY

Wavelet:
haar

Decomposition Level:
1

Sub-bands:
['LL', 'LH', 'HL', 'HH']

Features per sub-band:
['mean', 'std', 'variance', 'energy', 'entropy']

Total DWT features:
20

Feature names:
 - DWT_LL_mean
 - DWT_LL_std
 - DWT_LL_variance
 - DWT_LL_energy
 - DWT_LL_entropy
 - DWT_LH_mean
 - DWT_LH_std
 - DWT_LH_variance
 - DWT_LH_energy
 - DWT_LH_entropy
 - DWT_HL_mean
 - DWT_HL_std
 - DWT_HL_variance
 - DWT_HL_energy
 - DWT_HL_entropy
 - DWT_HH_mean
 - DWT_HH_std
 - DWT_HH_variance
 - DWT_HH_energy
 - DWT_HH_entropy


# EXTRACT DWT FEATURES FROM SAVED ROI DATASET

In [3]:
# ============================================================
# CELL 3: EXTRACT DWT FEATURES FROM SAVED ROI DATASET
# ============================================================

import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ============================================================
# FUNCTION: PROCESS ONE DATASET SPLIT
# ============================================================

def extract_features_from_split(
    split_path,
    split_name
):
    """
    Extract DWT features from all saved ROI images
    in a train or test split.
    """

    all_features = []

    # --------------------------------------------------------
    # Get class folders
    # --------------------------------------------------------

    class_folders = sorted([
        folder
        for folder in split_path.iterdir()
        if folder.is_dir()
    ])

    # --------------------------------------------------------
    # Process each class
    # --------------------------------------------------------

    for class_folder in class_folders:

        class_name = class_folder.name

        # ----------------------------------------------------
        # Get ROI image files
        # ----------------------------------------------------

        image_files = [
            file
            for file in class_folder.iterdir()
            if file.is_file()
            and file.suffix.lower() in [
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".tif",
                ".tiff"
            ]
        ]

        # ----------------------------------------------------
        # tqdm progress bar
        # ----------------------------------------------------

        progress_bar = tqdm(
            image_files,
            desc=f"{split_name} - {class_name}",
            unit="image"
        )

        # ----------------------------------------------------
        # Process every ROI
        # ----------------------------------------------------

        for image_path in progress_bar:

            try:

                # ------------------------------------------------
                # Read saved ROI
                # ------------------------------------------------

                roi = cv2.imread(
                    str(image_path),
                    cv2.IMREAD_GRAYSCALE
                )

                # ------------------------------------------------
                # Check whether image was read successfully
                # ------------------------------------------------

                if roi is None:

                    continue

                # ------------------------------------------------
                # Extract DWT features
                # ------------------------------------------------

                features = extract_dwt_features(
                    roi
                )

                # ------------------------------------------------
                # Handle empty ROI
                # ------------------------------------------------

                if features is None:

                    continue

                # ------------------------------------------------
                # Create one record
                # ------------------------------------------------

                record = {}

                # ------------------------------------------------
                # Add DWT features
                # ------------------------------------------------

                for index, feature_name in enumerate(
                    FEATURE_NAMES
                ):

                    record[
                        feature_name
                    ] = features[index]

                # ------------------------------------------------
                # Add class label
                # ------------------------------------------------

                record["class"] = class_name

                # ------------------------------------------------
                # Add original filename
                # ------------------------------------------------

                record["filename"] = image_path.name

                # ------------------------------------------------
                # Store record
                # ------------------------------------------------

                all_features.append(
                    record
                )

                # ------------------------------------------------
                # Update progress bar
                # ------------------------------------------------

                progress_bar.set_postfix(
                    extracted=len(all_features)
                )

            except Exception as e:

                print(
                    f"\nError processing "
                    f"{image_path.name}: {e}"
                )

    # --------------------------------------------------------
    # Convert to DataFrame
    # --------------------------------------------------------

    dataframe = pd.DataFrame(
        all_features
    )

    return dataframe


# ============================================================
# EXTRACT TRAIN FEATURES
# ============================================================

print("\n" + "=" * 60)
print("EXTRACTING DWT FEATURES - TRAIN")
print("=" * 60)

dwt_train_df = extract_features_from_split(
    ROI_TRAIN_PATH,
    "TRAIN"
)


# ============================================================
# EXTRACT TEST FEATURES
# ============================================================

print("\n" + "=" * 60)
print("EXTRACTING DWT FEATURES - TEST")
print("=" * 60)

dwt_test_df = extract_features_from_split(
    ROI_TEST_PATH,
    "TEST"
)


# ============================================================
# SAVE TRAIN CSV
# ============================================================

TRAIN_FEATURE_FILE = (
    FEATURE_PATH / "dwt_train.csv"
)

dwt_train_df.to_csv(
    TRAIN_FEATURE_FILE,
    index=False
)


# ============================================================
# SAVE TEST CSV
# ============================================================

TEST_FEATURE_FILE = (
    FEATURE_PATH / "dwt_test.csv"
)

dwt_test_df.to_csv(
    TEST_FEATURE_FILE,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("DWT FEATURE EXTRACTION COMPLETE")
print("=" * 60)

print(
    "\nTraining samples:",
    len(dwt_train_df)
)

print(
    "Testing samples:",
    len(dwt_test_df)
)

print(
    "\nNumber of DWT features:",
    len(FEATURE_NAMES)
)

print(
    "\nTraining CSV:",
    TRAIN_FEATURE_FILE
)

print(
    "Testing CSV:",
    TEST_FEATURE_FILE
)


EXTRACTING DWT FEATURES - TRAIN


TRAIN - glioma:   0%|          | 0/1108 [00:00<?, ?image/s]

TRAIN - meningioma:   0%|          | 0/1320 [00:00<?, ?image/s]

TRAIN - pituitary:   0%|          | 0/1455 [00:00<?, ?image/s]


EXTRACTING DWT FEATURES - TEST


TEST - glioma:   0%|          | 0/234 [00:00<?, ?image/s]

TEST - meningioma:   0%|          | 0/301 [00:00<?, ?image/s]

TEST - pituitary:   0%|          | 0/295 [00:00<?, ?image/s]


DWT FEATURE EXTRACTION COMPLETE

Training samples: 3883
Testing samples: 830

Number of DWT features: 20

Training CSV: C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_train.csv
Testing CSV: C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_test.csv


# STANDARDIZATION + PCA (DWT)

In [4]:
# ============================================================
# CELL 4: STANDARDIZATION + PCA (DWT)
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# 1. SEPARATE FEATURES AND LABELS
# ============================================================

# DWT feature columns
X_train = dwt_train_df[
    FEATURE_NAMES
].values

X_test = dwt_test_df[
    FEATURE_NAMES
].values


# Class labels
y_train = dwt_train_df[
    "class"
].values

y_test = dwt_test_df[
    "class"
].values


# ============================================================
# 2. STANDARDIZE FEATURES
# ============================================================
#
# DWT features have different numerical ranges.
# Standardization puts them on a comparable scale.
#
# IMPORTANT:
# Fit ONLY on training data.
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


# ============================================================
# 3. PCA
# ============================================================
#
# Reduce the DWT feature space to 15 components.
#
# Current DWT feature count:
# 20 features
#
# 20 DWT features → 15 PCA components
# ============================================================

pca = PCA(
    n_components=15,
    random_state=42
)


# Fit PCA ONLY on training data
X_train_pca = pca.fit_transform(
    X_train_scaled
)


# Apply the SAME PCA transformation to test data
X_test_pca = pca.transform(
    X_test_scaled
)


# ============================================================
# 4. PCA INFORMATION
# ============================================================

print("=" * 60)
print("DWT + PCA RESULTS")
print("=" * 60)

print(
    "\nOriginal number of features:",
    X_train.shape[1]
)

print(
    "Number of PCA components:",
    X_train_pca.shape[1]
)

print(
    "Training shape after PCA:",
    X_train_pca.shape
)

print(
    "Testing shape after PCA:",
    X_test_pca.shape
)

print(
    "\nTotal explained variance:",
    round(
        np.sum(
            pca.explained_variance_ratio_
        ) * 100,
        2
    ),
    "%"
)


# ============================================================
# 5. SHOW VARIANCE CONTRIBUTION
# ============================================================

print("\nVariance explained by each component:")

for i, variance in enumerate(
    pca.explained_variance_ratio_,
    start=1
):

    print(
        f"PC{i}: {variance * 100:.2f}%"
    )

DWT + PCA RESULTS

Original number of features: 20
Number of PCA components: 15
Training shape after PCA: (3883, 15)
Testing shape after PCA: (830, 15)

Total explained variance: 99.79 %

Variance explained by each component:
PC1: 34.36%
PC2: 26.51%
PC3: 12.15%
PC4: 5.51%
PC5: 5.29%
PC6: 4.82%
PC7: 4.55%
PC8: 2.68%
PC9: 2.02%
PC10: 0.85%
PC11: 0.40%
PC12: 0.27%
PC13: 0.14%
PC14: 0.13%
PC15: 0.10%


# SAVE DWT PCA FEATURES FOR SVM AND KNN

In [5]:
# ============================================================
# CELL 5: SAVE DWT PCA FEATURES FOR SVM AND KNN
# ============================================================

import pandas as pd


# ============================================================
# 1. CREATE PCA COLUMN NAMES
# ============================================================

pca_feature_names = [
    f"PC{i}"
    for i in range(
        1,
        X_train_pca.shape[1] + 1
    )
]


# ============================================================
# 2. CREATE TRAINING PCA DATAFRAME
# ============================================================

pca_train_df = pd.DataFrame(
    X_train_pca,
    columns=pca_feature_names
)

# Add class label
pca_train_df["class"] = y_train

# Add filename
pca_train_df["filename"] = (
    dwt_train_df["filename"].values
)


# ============================================================
# 3. CREATE TESTING PCA DATAFRAME
# ============================================================

pca_test_df = pd.DataFrame(
    X_test_pca,
    columns=pca_feature_names
)

# Add class label
pca_test_df["class"] = y_test

# Add filename
pca_test_df["filename"] = (
    dwt_test_df["filename"].values
)


# ============================================================
# 4. SAVE TRAIN PCA CSV
# ============================================================

PCA_TRAIN_FILE = (
    FEATURE_PATH / "dwt_pca_train.csv"
)

pca_train_df.to_csv(
    PCA_TRAIN_FILE,
    index=False
)


# ============================================================
# 5. SAVE TEST PCA CSV
# ============================================================

PCA_TEST_FILE = (
    FEATURE_PATH / "dwt_pca_test.csv"
)

pca_test_df.to_csv(
    PCA_TEST_FILE,
    index=False
)


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("=" * 60)
print("DWT PCA DATASET SAVED")
print("=" * 60)

print(
    "\nTraining samples:",
    len(pca_train_df)
)

print(
    "Testing samples:",
    len(pca_test_df)
)

print(
    "PCA components:",
    len(pca_feature_names)
)

print(
    "\nTraining CSV:"
)

print(PCA_TRAIN_FILE)

print(
    "\nTesting CSV:"
)

print(PCA_TEST_FILE)


# ============================================================
# 7. SHOW FIRST FEW ROWS
# ============================================================

print("\nFirst 5 training samples:")

display(
    pca_train_df.head()
)

DWT PCA DATASET SAVED

Training samples: 3883
Testing samples: 830
PCA components: 15

Training CSV:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_pca_train.csv

Testing CSV:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_pca_test.csv

First 5 training samples:


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,class,filename
0,4.348393,-1.518085,-3.879745,-0.422785,-0.499384,-0.138510,-0.030534,0.116147,-0.584509,0.039308,0.183875,-0.316562,0.050242,0.257271,-0.377596,glioma,brisc2025_train_00001_gl_ax_t1.jpg
1,5.452324,4.839042,-3.408530,-1.145836,0.205934,-0.689416,0.886037,0.435759,1.056247,-0.843423,0.356602,-0.989911,-0.302035,0.432650,0.231123,glioma,brisc2025_train_00002_gl_ax_t1.jpg
2,-0.645747,-0.284993,1.540573,-0.058573,-0.877807,-0.499421,2.660609,-2.053974,0.354101,0.130983,0.302587,0.016473,-0.190245,0.410798,-0.026479,glioma,brisc2025_train_00003_gl_ax_t1.jpg
3,-0.893311,-2.205389,0.395334,0.545549,-0.920797,0.354761,0.347853,0.175217,0.157088,-0.217770,-0.270235,-0.033699,0.036263,0.018218,-0.036260,glioma,brisc2025_train_00004_gl_ax_t1.jpg
4,-0.137490,-0.071099,-1.242021,0.741794,-0.890952,1.445732,0.001708,0.071303,-0.366765,-0.012545,-0.072237,0.028479,-0.134699,-0.108302,-0.052076,glioma,brisc2025_train_00005_gl_ax_t1.jpg


#  SAVE TRAINED DWT SCALER AND PCA

In [6]:
# ============================================================
# CELL 6: SAVE TRAINED DWT SCALER AND PCA
# ============================================================

import joblib


# ============================================================
# 1. SAVE STANDARD SCALER
# ============================================================

SCALER_FILE = (
    FEATURE_PATH / "dwt_scaler.pkl"
)

joblib.dump(
    scaler,
    SCALER_FILE
)


# ============================================================
# 2. SAVE PCA MODEL
# ============================================================

PCA_FILE = (
    FEATURE_PATH / "dwt_pca.pkl"
)

joblib.dump(
    pca,
    PCA_FILE
)


# ============================================================
# 3. DISPLAY SAVED FILES
# ============================================================

print("=" * 60)
print("DWT SCALER AND PCA SAVED")
print("=" * 60)

print("\nScaler saved at:")
print(SCALER_FILE)

print("\nPCA saved at:")
print(PCA_FILE)

print("\nPCA components:", pca.n_components_)

print(
    "Explained variance:",
    round(
        np.sum(
            pca.explained_variance_ratio_
        ) * 100,
        2
    ),
    "%"
)

DWT SCALER AND PCA SAVED

Scaler saved at:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_scaler.pkl

PCA saved at:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_pca.pkl

PCA components: 15
Explained variance: 99.79 %
